# Task 3
Assesing the provided EQ profile instance of the CGMES model.

In [1]:
# Standard library: for building a filesystem path to the CGMES XML file
from pathlib import Path

# pandas: used throughout the notebook to hold and query model data as tables
import pandas as pd
# triplets: parses a CIM/CGMES RDF-XML file into a flat "triples" table,
# where every fact in the model becomes one row of (ID, KEY, VALUE)
import triplets


### Task 3.1.
First we import the  provided EQ profile and asses the total production capacity of the generators in the model.

In [2]:
# Load the EQ profile XML file into a "triples" table.
# Every CIM object in the file becomes a set of rows sharing the same ID:
#   ID    -> the RDF identifier of the object (the "subject")
#   KEY   -> the CIM attribute/relationship name (the "predicate")
#   VALUE -> the attribute value or the ID of the referenced object (the "object")
model_path = Path.cwd() / "20210325T1530Z_1D_NL_EQ_001 3.xml"
triples = triplets.parse(str(model_path), return_type="pandas")

# Step 1: find every object whose CIM class ("Type") is "GeneratingUnit".
# A GeneratingUnit is the CIM object that carries a generator's power limits
# (minimum/nominal/maximum active power), as opposed to the SynchronousMachine
# object, which carries the electrical machine parameters.
generator_ids = set(
    triples.loc[
        triples["VALUE"].eq("GeneratingUnit"),
        "ID",
    ]
)
# Keep only the rows that belong to those GeneratingUnit objects.
generator_properties = triples[triples["ID"].isin(generator_ids)]

# Small helper: for a given CIM property name, return a Series indexed by
# object ID -> property value (one row per object).
def property_values(property_name):
    values = generator_properties[generator_properties["KEY"].eq(property_name)]
    return values.set_index("ID")["VALUE"]

# Step 2: pull out the specific properties we need for each generating unit.
names = property_values("IdentifiedObject.name")
mrids = property_values("IdentifiedObject.mRID")
minimum_power = property_values("GeneratingUnit.minOperatingP").astype(float)
nominal_power = property_values("GeneratingUnit.nominalP").astype(float)
maximum_power = property_values("GeneratingUnit.maxOperatingP").astype(float)

# Step 3: assemble everything into one readable summary table.
generators = pd.DataFrame(
    {
        "generator": names,
        "mRID": mrids,
        "minimum_operating_power_MW": minimum_power,
        "nominal_power_MW": nominal_power,
        "maximum_operating_power_MW": maximum_power,
    }
).sort_values("generator").reset_index(drop=True)

display(generators)

# Step 4: the model's total production capacity is simply the sum of each
# generator's own capacity - i.e. add up one column at a time.
maximum_capacity_mw = generators["maximum_operating_power_MW"].sum()
nominal_capacity_mw = generators["nominal_power_MW"].sum()
minimum_capacity_mw = generators["minimum_operating_power_MW"].sum()

print(f"Total maximum production capacity: {maximum_capacity_mw:.0f} MW")
print(f"Total nominal production capacity: {nominal_capacity_mw:.0f} MW")
print(f"Total minimum operating power: {minimum_capacity_mw:.0f} MW")


,generator,mRID,minimum_operating_power_MW,nominal_power_MW,maximum_operating_power_MW
0,Gen-12908,b850063d-eae7-4675-bc98-4642d3076783,130.0,225.0,250.0
1,Gen-12910,ca80ee09-3bed-4884-bc28-6dc89d067289,130.0,225.0,250.0
2,Gen-12923,049438a6-780a-44fe-a788-ebe385d98e25,300.0,990.0,1000.0


Total maximum production capacity: 1500 MW
Total nominal production capacity: 1440 MW
Total minimum operating power: 560 MW


### Task 3.2.
Now we asses the nominal voltages of the windins of the transformer NL_TR2_2 (ID:_2184f365-8cd5-4b5d-8a28-9d68603bb6a4)

In [3]:
# The transformer we are asked about, NL_TR2_2, identified by its mRID.
target_transformer_id = "2184f365-8cd5-4b5d-8a28-9d68603bb6a4"

# CGMES/RDF identifiers are written in two slightly different ways in this
# file: as an "rdf:ID" attribute (e.g. "_2184f365-...") and as a plain
# "IdentifiedObject.mRID" value (e.g. "2184f365-..."), and references to an
# object elsewhere use a leading "#" (e.g. "#_2184f365-..."). This helper
# strips those leading characters so all three forms can be compared.
def normalize_id(value):
    return str(value).lstrip("#_")

# A transformer's electrical windings are modelled as separate
# "PowerTransformerEnd" objects (one per winding/voltage level), each of
# which points back to its parent "PowerTransformer" via the
# "PowerTransformerEnd.PowerTransformer" relationship.
# Step 1: find every PowerTransformerEnd object in the model.
end_type_ids = set(
    triples.loc[
        triples["VALUE"].eq("PowerTransformerEnd"),
        "ID",
    ]
)
end_properties = triples[triples["ID"].isin(end_type_ids)].copy()

# Step 2: keep only the ends that explicitly reference NL_TR2_2.
linked_end_ids = set(
    end_properties.loc[
        end_properties["KEY"].eq("PowerTransformerEnd.PowerTransformer")
        & end_properties["VALUE"].map(normalize_id).eq(target_transformer_id),
        "ID",
    ]
)

if not linked_end_ids:
    raise ValueError(f"No transformer ends reference transformer {target_transformer_id}.")

transformer_end_properties = end_properties[
    end_properties["ID"].isin(linked_end_ids)
]

# Small helper: some IDs below turn out to carry more than one distinct
# value for the same property (see the note after this cell) - join all of
# them together so nothing is silently hidden.
def all_values(properties, property_name):
    values = properties.loc[properties["KEY"].eq(property_name), "VALUE"]
    return "; ".join(sorted({str(value) for value in values}))

# Step 3: build one row per winding with its rated (nominal) voltage.
winding_rows = []
for end_id, properties in transformer_end_properties.groupby("ID"):
    winding_rows.append(
        {
            "transformer_end_id": end_id,
            "name": all_values(properties, "IdentifiedObject.name"),
            "end_number": all_values(properties, "TransformerEnd.endNumber"),
            "rated_voltage_kV": all_values(properties, "PowerTransformerEnd.ratedU"),
            "rated_apparent_power_MVA": all_values(properties, "PowerTransformerEnd.ratedS"),
        }
    )

windings = pd.DataFrame(winding_rows).sort_values(
    ["end_number", "transformer_end_id"]
).reset_index(drop=True)
display(windings)

print(f"Transformer: NL_TR2_2 ({target_transformer_id})")
print("Nominal winding voltages: " + ", ".join(
    f"{row.end_number}: {row.rated_voltage_kV} kV"
    for row in windings.itertuples()
))


,transformer_end_id,name,end_number,rated_voltage_kV,rated_apparent_power_MVA
0,0dbed103-fc51-4df4-a6fa-0dee4c57f3a3,NL_TR2_2; NL_TR2_3; NL_TR2_4,1,220,1260
1,41ca9e70-1cb4-4971-b8e6-a97a15580b89,NL_TR2_2,2,15.75,1260


Transformer: NL_TR2_2 (2184f365-8cd5-4b5d-8a28-9d68603bb6a4)
Nominal winding voltages: 1: 220 kV, 2: 15.75 kV


The results of this task also exposes a mistake in the model, the first winding of the transformer NL_TR2_2 has a duplicate-ID issue, which would not be allowed as RDF identifiers and CGMES mRID values must uniquely identify one object. Reusing them means the XML contains multiple distinct objects with the same identity.

### Task 3.3.
We investigate the permanently and temporarily allowed limit for line segment NL-Line_5 (ID: _e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc)

In [8]:
# The line segment we are asked about, NL-Line_5, identified by its mRID.
target_line_id = "e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc"
target_line_name = "NL-Line_5"

# Normalize references because triplets removes the RDF '#' fragment and
# the original rdf:ID uses a leading underscore.
def normalize_id(value):
    return str(value).lstrip("#_")

line_properties = triples[triples["ID"].map(normalize_id).eq(target_line_id)]
if line_properties.empty:
    raise ValueError(f"Line {target_line_name} ({target_line_id}) was not found.")

line_name = line_properties.loc[
    line_properties["KEY"].eq("IdentifiedObject.name"),
    "VALUE",
].iloc[0]

# Current limits (PATL/TATL) are not stored directly on the line. Instead,
# CGMES links them through a small chain of objects:
# ACLineSegment -> Terminal -> OperationalLimitSet -> OperationalLimit
# so we have to walk that chain step by step.

# Step 1: find the terminals whose conducting equipment is NL-Line_5.
line_terminal_ids = set(
    triples.loc[
        triples["KEY"].eq("Terminal.ConductingEquipment")
        & triples["VALUE"].map(normalize_id).eq(target_line_id),
        "ID",
    ]
)

# Step 2: map each operational-limit set to the terminal it belongs to.
limit_set_terminal_rows = triples[
    triples["KEY"].eq("OperationalLimitSet.Terminal")
    & triples["ID"].isin(
        set(
            triples.loc[
                triples["KEY"].eq("OperationalLimitSet.Terminal")
                & triples["VALUE"].isin(line_terminal_ids),
                "ID",
            ]
        )
    )
]
line_limit_set_ids = set(limit_set_terminal_rows["ID"])
limit_set_to_terminal = dict(
    zip(limit_set_terminal_rows["ID"], limit_set_terminal_rows["VALUE"])
)

# Step 3: find the individual operational limits that belong to those sets.
line_limit_ids = set(
    triples.loc[
        triples["KEY"].eq("OperationalLimit.OperationalLimitSet")
        & triples["VALUE"].isin(line_limit_set_ids),
        "ID",
    ]
)
limit_properties = triples[triples["ID"].isin(line_limit_ids)]

# Step 4: each limit references an OperationalLimitType, which tells us
# whether it is a PATL (permanently allowed) or TATL (temporarily allowed)
# limit and for how long it may be applied.
limit_type_ids = set(
    limit_properties.loc[
        limit_properties["KEY"].eq("OperationalLimit.OperationalLimitType"),
        "VALUE",
    ]
)
limit_type_properties = triples[triples["ID"].isin(limit_type_ids)]

kind_by_type = {}
for limit_type_id, properties in limit_type_properties.groupby("ID"):
    kind = properties.loc[
        properties["KEY"].eq("OperationalLimitType.kind"),
        "VALUE",
    ]
    duration = properties.loc[
        properties["KEY"].eq("OperationalLimitType.acceptableDuration"),
        "VALUE",
    ]
    kind_by_type[limit_type_id] = {
        "limit_kind": kind.iloc[0] if not kind.empty else None,
        "acceptable_duration_s": duration.iloc[0] if not duration.empty else None,
    }

# Step 5: combine everything into one table: which terminal, which kind of
# limit (PATL/TATL), and the current value in amps.
limit_rows = []
for limit_id, properties in limit_properties.groupby("ID"):
    limit_type_id = properties.loc[
        properties["KEY"].eq("OperationalLimit.OperationalLimitType"),
        "VALUE",
    ].iloc[0]
    limit_set_id = properties.loc[
        properties["KEY"].eq("OperationalLimit.OperationalLimitSet"),
        "VALUE",
    ].iloc[0]
    value = float(
        properties.loc[
            properties["KEY"].eq("CurrentLimit.normalValue"),
            "VALUE",
        ].iloc[0]
    )
    type_data = kind_by_type[limit_type_id]
    limit_rows.append(
        {
            "terminal_id": limit_set_to_terminal[limit_set_id],
            "limit_kind": type_data["limit_kind"],
            "acceptable_duration_s": type_data["acceptable_duration_s"],
            "current_limit_A": value,
        }
    )

limits = pd.DataFrame(limit_rows).sort_values(
    ["terminal_id", "limit_kind"]
).reset_index(drop=True)
display(limits)

permanent_limits = limits.loc[
    limits["limit_kind"].eq("LimitKind.patl"),
    "current_limit_A",
]
temporary_limits = limits.loc[
    limits["limit_kind"].eq("LimitKind.tatl"),
    "current_limit_A",
]

if permanent_limits.empty or temporary_limits.empty:
    raise ValueError("Both permanent and temporary limits were not found.")

permanent_limit_a = permanent_limits.iloc[0]
temporary_limit_a = temporary_limits.iloc[0]
print(f"Line: {line_name} ({target_line_id})")
print(f"Permanent allowed limit (PATL): {permanent_limit_a:.0f} A")
print(f"Temporary allowed limit (TATL): {temporary_limit_a:.0f} A")
print(f"Temporary limit duration: 600 s")


,terminal_id,limit_kind,acceptable_duration_s,current_limit_A
0,757d4f50-707b-47a0-891c-cbaefd649631,LimitKind.patl,None,1876.0
1,757d4f50-707b-47a0-891c-cbaefd649631,LimitKind.tatl,600,500.0
2,ae588863-b154-451d-978a-7ab08ac50fb6,LimitKind.patl,None,1876.0
3,ae588863-b154-451d-978a-7ab08ac50fb6,LimitKind.tatl,600,500.0


Line: NL-Line_5 (e8acf6b6-99cb-45ad-b8dc-16c7866a4ddc)
Permanent allowed limit (PATL): 1876 A
Temporary allowed limit (TATL): 500 A
Temporary limit duration: 600 s


The permanently allowed limit (PATL) is 1876A. and temporarily allowed limit (TATL) is 500A for this line segment. 

This again seems like an error in the model, as the TATL should be higher than the PATL. 500 A TATL vs 1876 A PATL doesn't make physical sense. A 10-minute overload rating (TATL) should always be higher or equal to the continuous rating (PATL), often noticeably higher, commonly 1.1–1.5× or more, depending on the thermal rating methodology. Having TATL at roughly a quarter of PATL is not a legitimate operating limit relationship. By the logic in this model the equipment is only allowed to carry more current continuously than it's allowed to carry briefly under overload, which is backwards.

The conceptual difference between PATL and TATL is that PATL is the continuous rating, the maximum current the line can carry indefinitely without overheating, while TATL is the short-term overload rating, the maximum current the line can carry for a limited time (in this case 10 minutes) before it must be reduced to avoid damage. And the mathmatical difference doesn't make much sense here as the TATL is lower than the PATL, which is not physically possible.

Knowing there is an issue with the TATL value with this one line, we can check the other lines in the model to see if this is a recurring issue. The following code checks all line segments in the model and prints all PATL and TATL values for each line segment.

In [5]:
# Audit every PATL and TATL value associated with EVERY line segment in the
# model (not just NL-Line_5), so we can check whether the odd PATL/TATL
# relationship we found for NL-Line_5 is a one-off or a model-wide problem.

# Small helper: return the first value found for a given property, or None
# if the object does not have that property at all.
def first_value(rows, key):
    values = rows.loc[rows["KEY"].eq(key), "VALUE"]
    return values.iloc[0] if not values.empty else None

# Step 1: look up every object's human-readable name and CIM class ("Type")
# once, so we can label results without repeating lookups later.
object_names = {}
object_types = {}
for object_id, properties in triples.groupby("ID"):
    object_names[object_id] = first_value(properties, "IdentifiedObject.name")
    object_types[object_id] = first_value(properties, "Type")

# Step 2: collect the IDs of all ACLineSegment (i.e. power line) objects.
line_ids = {
    object_id
    for object_id, object_type in object_types.items()
    if object_type == "ACLineSegment"
}

# Step 3: collect the OperationalLimitType objects that represent a PATL or
# a TATL, together with their acceptable duration.
limit_type_ids = set(
    triples.loc[
        triples["KEY"].eq("OperationalLimitType.kind")
        & triples["VALUE"].isin(["LimitKind.patl", "LimitKind.tatl"]),
        "ID",
    ]
)
limit_type_data = {}
for limit_type_id in limit_type_ids:
    properties = triples[triples["ID"].eq(limit_type_id)]
    limit_type_data[limit_type_id] = {
        "limit_kind": first_value(properties, "OperationalLimitType.kind"),
        "acceptable_duration_s": first_value(
            properties, "OperationalLimitType.acceptableDuration"
        ),
        "is_infinite_duration": first_value(
            properties, "OperationalLimitType.isInfiniteDuration"
        ),
    }

# Step 4: walk the same ACLineSegment -> Terminal -> OperationalLimitSet
# chain as before, but this time for all lines at once.
line_terminal_rows = triples[
    triples["KEY"].eq("Terminal.ConductingEquipment")
    & triples["VALUE"].isin(line_ids)
]
line_terminal_ids = set(line_terminal_rows["ID"])
terminal_to_line = dict(zip(line_terminal_rows["ID"], line_terminal_rows["VALUE"]))

limit_set_rows = triples[
    triples["KEY"].eq("OperationalLimitSet.Terminal")
    & triples["VALUE"].isin(line_terminal_ids)
]
limit_set_to_terminal = dict(zip(limit_set_rows["ID"], limit_set_rows["VALUE"]))

# Step 5: assemble one row per (line, terminal, PATL-or-TATL) combination.
limit_rows = []
for limit_id, properties in triples.groupby("ID"):
    limit_type_id = first_value(properties, "OperationalLimit.OperationalLimitType")
    limit_set_id = first_value(properties, "OperationalLimit.OperationalLimitSet")
    value = first_value(properties, "CurrentLimit.normalValue")
    if (
        limit_type_id not in limit_type_ids
        or limit_set_id not in limit_set_to_terminal
        or value is None
    ):
        continue

    terminal_id = limit_set_to_terminal[limit_set_id]
    line_id = terminal_to_line[terminal_id]
    type_data = limit_type_data[limit_type_id]
    limit_rows.append(
        {
            "line": object_names.get(line_id, line_id),
            "terminal_id": terminal_id,
            "limit_kind": type_data["limit_kind"],
            "current_limit_A": float(value),
            "acceptable_duration_s": type_data["acceptable_duration_s"],
            "is_infinite_duration": type_data["is_infinite_duration"],
        }
    )

line_current_limits = pd.DataFrame(limit_rows).sort_values(
    ["line", "terminal_id", "limit_kind"]
).reset_index(drop=True)
display(line_current_limits)

print(f"Line PATL records: {line_current_limits['limit_kind'].eq('LimitKind.patl').sum()}")
print(f"Line TATL records: {line_current_limits['limit_kind'].eq('LimitKind.tatl').sum()}")


,line,terminal_id,limit_kind,current_limit_A,acceptable_duration_s,is_infinite_duration
0,NL-Line_1,357e3e14-c38e-4a7e-9c95-b3dbe158f5f3,LimitKind.patl,1233.9,None,true
1,NL-Line_1,357e3e14-c38e-4a7e-9c95-b3dbe158f5f3,LimitKind.tatl,500.0,600,false
2,NL-Line_1,6b1cd30c-19ba-44e1-9447-d01db6b1ef9d,LimitKind.patl,1233.9,None,true
3,NL-Line_1,6b1cd30c-19ba-44e1-9447-d01db6b1ef9d,LimitKind.tatl,500.0,600,false
4,NL-Line_2,8f308117-8bbd-4798-b145-4e78f7d049e7,LimitKind.patl,1371.0,None,true
5,NL-Line_2,8f308117-8bbd-4798-b145-4e78f7d049e7,LimitKind.tatl,500.0,600,false
6,NL-Line_2,c557146e-dfc4-4020-9738-d592b188338b,LimitKind.patl,1371.0,None,true
7,NL-Line_2,c557146e-dfc4-4020-9738-d592b188338b,LimitKind.tatl,500.0,600,false
8,NL-Line_3,5dfee914-a4fd-4bde-a5b4-c4caa6378d10,LimitKind.patl,2000.0,None,true
9,NL-Line_3,5dfee914-a4fd-4bde-a5b4-c4caa6378d10,LimitKind.tatl,500.0,600,false


Line PATL records: 10
Line TATL records: 10


As can be seen with the help of the diagnostic code above, all TATL values in the model are set to 500 A, while each PATL is unique to its associated line (Both terminals of the line have the same values), which means that the TATL values are an error in the model.

This is likely a model error due to the fact that every single TATL value is set to 500 A regardless of the associated equipment's actual current rating. Additionally, inspecting the per unit resistance values of the ACLineSegment.r for NL-Line_5 seem consistant with a bundeled high voltage power line.

### Task 3.4.
We investigate which generator is set as slack in the given model. 

Power grid models typically designate one generator as the slack bus, which serves as the reference point for voltage in the system. More specifically, the slack bus establishes a voltage angle reference (usually set to 0 degrees), which a solver needs in order to converge on a single solution during power flow analysis. The slack generator also compensates for any imbalance between generation and load, both active and reactive power, ensuring that the system remains balanced.

A physcial analogy to this would be a connection point to a large, stiff grid, which can absorb fluctuations in power without significant changes in voltage or frequency. In practice, the slack generator is often a large conventional power plant, such as a gas power plant, which has the capability to adjust its output quickly to maintain system stability.

In [6]:
# Inspect generator and control attributes that may indicate a slack candidate.
# A generator is a good slack candidate if it explicitly regulates voltage
# (i.e. it has a RegulatingControl in "voltage" mode), since that is the
# behaviour a slack/reference bus needs to provide during a power-flow study.
type_rows = triples[triples["KEY"].eq("Type")]

def normalize_id(value):
    return str(value).lstrip("#_")

# Small helper: build a plain {property_name: value} dictionary for one
# object, so we can look up several properties at once without repeated
# dataframe filtering.
def property_map(object_id):
    rows = triples[triples["ID"].eq(object_id)]
    return dict(zip(rows["KEY"], rows["VALUE"]))

def first_property(properties, *names):
    for name in names:
        if name in properties:
            return properties[name]
    return None

# Step 1: find every SynchronousMachine object (i.e. every generator).
machine_ids = set(
    type_rows.loc[
        type_rows["VALUE"].eq("SynchronousMachine"),
        "ID",
    ]
)
machine_rows = []
for machine_id in machine_ids:
    properties = property_map(machine_id)
    control_reference = first_property(
        properties,
        "RegulatingCondEq.RegulatingControl",
    )
    machine_rows.append(
        {
            "machine_id": machine_id,
            "generator_name": properties.get("IdentifiedObject.name"),
            "generating_unit_id": first_property(
                properties,
                "RotatingMachine.GeneratingUnit",
            ),
            "rated_voltage_kV": properties.get("RotatingMachine.ratedU"),
            "rated_power_MVA": properties.get("RotatingMachine.ratedS"),
            "min_reactive_power_Mvar": properties.get("SynchronousMachine.minQ"),
            "max_reactive_power_Mvar": properties.get("SynchronousMachine.maxQ"),
            "regulating_control_id": control_reference,
            "has_voltage_regulating_control": control_reference is not None,
        }
    )

generator_controls = pd.DataFrame(machine_rows)

# Step 2: add the generating-unit name for readability.
generating_unit_name_by_id = {}
for unit_id in set(generator_controls["generating_unit_id"].dropna()):
    generating_unit_name_by_id[unit_id] = property_map(unit_id).get(
        "IdentifiedObject.name"
    )
generator_controls["generating_unit_name"] = generator_controls[
    "generating_unit_id"
].map(generating_unit_name_by_id)

# Step 3: for machines that do have a regulating control, pull out its name,
# mode (e.g. voltage regulation) and controlled terminal.
def control_details(control_id):
    if not control_id:
        return {
            "control_name": None,
            "control_mode": None,
            "control_terminal": None,
        }
    properties = property_map(control_id)
    return {
        "control_name": properties.get("IdentifiedObject.name"),
        "control_mode": properties.get("RegulatingControl.mode"),
        "control_terminal": properties.get("RegulatingControl.Terminal"),
    }

control_data = generator_controls["regulating_control_id"].map(control_details)
control_data = pd.DataFrame(control_data.tolist(), index=generator_controls.index)
generator_controls = pd.concat([generator_controls, control_data], axis=1)
generator_controls = generator_controls.sort_values("generator_name").reset_index(drop=True)
display(generator_controls)

voltage_regulated = generator_controls[
    generator_controls["has_voltage_regulating_control"]
]
print("Generators with explicit voltage-regulating control:")
print(", ".join(voltage_regulated["generator_name"].dropna()) or "None")


,machine_id,generator_name,generating_unit_id,rated_voltage_kV,...,generating_unit_name,control_name,control_mode,control_terminal
0,9c3b8f97-7972-477d-9dc8-87365cc0ad0e,NL-G1,049438a6-780a-44fe-a788-ebe385d98e25,15.75,...,Gen-12923,NL-G1,RegulatingControlModeKind.voltage,faab7959-f9bf-421b-bc3f-d364e0c1388b
1,2844585c-0d35-488d-a449-685bcd57afbf,NL-G2,ca80ee09-3bed-4884-bc28-6dc89d067289,15.75,...,Gen-12910,None,None,None
2,1dc9afba-23b5-41a0-8540-b479ed8baf4b,NL-G3,b850063d-eae7-4675-bc98-4642d3076783,None,...,Gen-12908,None,None,None


Generators with explicit voltage-regulating control:
NL-G1


As only one of the generators present in this EQ profile has voltage regulating control enabled, it is reasonable to conclude that this generator is the slack generator in the model. The generator with voltage control enabled is NL-G1 (rdf:ID="_9c3b8f97-7972-477d-9dc8-87365cc0ad0e).

### Task 3.5.
We compile the mistakes already surfaced in Tasks 3.2 and 3.3, manually check some additional aspects in the XML file, and then run a few extra automated checks over the whole model to look for other, less obvious errors. The task description specifically asks for semantic, power-system, and logical errors, so the checks below are grouped under those three headings.

**Note on tooling:** the cell below re-uses the `triples` table already loaded in Task 3.1,
so it must be run after that cell. It only needs `pandas`, which is already imported.


In [ ]:
def normalize_id(value):
    return str(value).lstrip("#_")

def first_value(rows, key):
    values = rows.loc[rows["KEY"].eq(key), "VALUE"]
    return values.iloc[0] if not values.empty else None


# 1) Duplicate rdf:ID check ---------------------------------------------------
# Every rdf:ID in a CGMES file must belong to exactly ONE object. If the same
# ID shows up with two *different* values for the SAME property (e.g. two
# different names), that proves several distinct objects were saved under one
# shared ID. triplets silently merges such rows (see the "NL_TR2_2; NL_TR2_3;
# NL_TR2_4" name in the Task 3.2 output above) instead of raising an error,
# so we check for this explicitly here.
duplicate_id_rows = []
for (object_id, key), group in triples.groupby(["ID", "KEY"]):
    distinct_values = group["VALUE"].unique()
    if len(distinct_values) > 1:
        duplicate_id_rows.append(
            {
                "ID": object_id,
                "property": key,
                "conflicting_values": "; ".join(map(str, distinct_values)),
            }
        )

duplicate_ids = pd.DataFrame(duplicate_id_rows)
print("1) Objects sharing one rdf:ID with conflicting property values:")
display(duplicate_ids)


# 2) Dangling / broken reference check for BaseVoltage -----------------------
# Most BaseVoltage objects (e.g. the 400 kV / 220 kV network voltages) are
# legitimately defined in the separate EQ *boundary* profile this model
# depends on, so a reference to them will not resolve locally - that is
# normal and expected. The 15.75 kV generator-transformer BaseVoltage,
# however, IS defined locally in this file, so every local reference that
# is supposed to point to it must resolve to that same, locally-defined ID.
local_base_voltage_ids = set(
    triples.loc[triples["VALUE"].eq("BaseVoltage"), "ID"].map(normalize_id)
)

base_voltage_refs = triples.loc[
    triples["KEY"].eq("TransformerEnd.BaseVoltage"), ["ID", "VALUE"]
].copy()
base_voltage_refs["normalized_target"] = base_voltage_refs["VALUE"].map(normalize_id)

# A boundary voltage is referenced many times across the file (once per piece
# of equipment at that voltage level); a genuinely broken reference is not.
reference_counts = base_voltage_refs["normalized_target"].value_counts()
boundary_voltage_ids = set(reference_counts[reference_counts > 1].index)
resolvable_ids = local_base_voltage_ids | boundary_voltage_ids

unresolved = base_voltage_refs[~base_voltage_refs["normalized_target"].isin(resolvable_ids)]
print("\n2) TransformerEnd.BaseVoltage references that do not resolve to any")
print("   locally-defined or repeatedly-used (boundary) BaseVoltage object:")
display(unresolved)


# 3) Missing mandatory attribute check for generators -------------------------
# Every SynchronousMachine should declare its rated voltage
# (RotatingMachine.ratedU); without it, load-flow/short-circuit tools cannot
# properly initialise the machine.
machine_ids = set(triples.loc[triples["VALUE"].eq("SynchronousMachine"), "ID"])
has_rated_u = set(
    triples.loc[
        triples["ID"].isin(machine_ids) & triples["KEY"].eq("RotatingMachine.ratedU"),
        "ID",
    ]
)
missing_rated_u_ids = machine_ids - has_rated_u
missing_rated_u = triples.loc[
    triples["ID"].isin(missing_rated_u_ids) & triples["KEY"].eq("IdentifiedObject.name"),
    ["ID", "VALUE"],
].rename(columns={"VALUE": "generator_name"})
print("\n3) Generators (SynchronousMachine) with no RotatingMachine.ratedU:")
display(missing_rated_u)


# 4) Generator active-power rating vs. apparent power & power factor ---------
# A generator cannot deliver more active power than ratedS x ratedPowerFactor.
# We recompute that theoretical maximum and compare it to the declared
# GeneratingUnit.maxOperatingP for every generator in the model.
def property_map(object_id):
    rows = triples[triples["ID"].eq(object_id)]
    return dict(zip(rows["KEY"], rows["VALUE"]))

rating_rows = []
for machine_id in machine_ids:
    machine_properties = property_map(machine_id)
    unit_id = machine_properties.get("RotatingMachine.GeneratingUnit")
    if not unit_id:
        continue
    unit_properties = property_map(unit_id)

    rated_s = machine_properties.get("RotatingMachine.ratedS")
    rated_pf = machine_properties.get("RotatingMachine.ratedPowerFactor")
    max_operating_p = unit_properties.get("GeneratingUnit.maxOperatingP")
    if rated_s is None or rated_pf is None or max_operating_p is None:
        continue

    rated_s, rated_pf, max_operating_p = float(rated_s), float(rated_pf), float(max_operating_p)
    theoretical_max_p = rated_s * rated_pf
    rating_rows.append(
        {
            "generator": machine_properties.get("IdentifiedObject.name"),
            "ratedS_MVA": rated_s,
            "ratedPowerFactor": rated_pf,
            "theoretical_max_P_MW": theoretical_max_p,
            "declared_maxOperatingP_MW": max_operating_p,
            "exceeds_theoretical_max": max_operating_p > theoretical_max_p,
        }
    )

rating_check = pd.DataFrame(rating_rows).sort_values("generator").reset_index(drop=True)
print("\n4) Declared maxOperatingP vs. ratedS x ratedPowerFactor:")
display(rating_check)


C:\Users\Kristoferis\AppData\Local\Temp\ipykernel_10432\2682240598.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (object_id, key), group in triples.groupby(["ID", "KEY"]):


1) Objects sharing one rdf:ID with conflicting property values:


,ID,property,conflicting_values
0,0dbed103-fc51-4df4-a6fa-0dee4c57f3a3,IdentifiedObject.name,NL_TR2_2; NL_TR2_3; NL_TR2_4
1,87da6373-3b6c-47a2-9493-1918a8d9df61,Model.profile,http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0; http://iec.ch/TC57/ns/CIM/ShortCircuit-EU/3.0



2) TransformerEnd.BaseVoltage references that do not resolve to any
   locally-defined or repeatedly-used (boundary) BaseVoltage object:


,ID,VALUE,normalized_target
1058,c614f321-2c88-4fa3-b98a-8ec3cde73db4,597e44dc-2a6e-4c62-82f3-f82cc46e0e14,597e44dc-2a6e-4c62-82f3-f82cc46e0e14
1167,4df90339-ae02-4da1-8e1e-69ab068df065,abb2348a-aa10-446f-9f5d-49f93f211534,abb2348a-aa10-446f-9f5d-49f93f211534



3) Generators (SynchronousMachine) with no RotatingMachine.ratedU:


,ID,generator_name
487,1dc9afba-23b5-41a0-8540-b479ed8baf4b,NL-G3



4) Declared maxOperatingP vs. ratedS x ratedPowerFactor:


,generator,ratedS_MVA,ratedPowerFactor,theoretical_max_P_MW,declared_maxOperatingP_MW,exceeds_theoretical_max
0,NL-G1,1100.0,0.9,990.0,1000.0,True
1,NL-G2,250.0,0.9,225.0,250.0,True
2,NL-G3,250.0,0.9,225.0,250.0,True


#### Compiled list of mistakes found in the model

**Semantic ** — these break the basic rules that every CGMES model must follow, independent of any power-system meaning:

1. **Duplicate rdf:ID reused for three different objects.** As noted in the Task 3.2, the ID `_0dbed103-fc51-4df4-a6fa-0dee4c57f3a3`
   is assigned to three separate `PowerTransformerEnd` objects, named `NL_TR2_2`, `NL_TR2_3`and `NL_TR2_4`. RDF/CGMES identifiers must uniquely identify one object; reusing an ID means the file actually contains multiple distinct objects claiming the same identity. All three copies also incorrectly point to the same Terminal and the same parent `PowerTransformer` (the real `NL_TR2_2`), even though two of them are labelled as if they belonged to different transformers.
   - The `NL_TR2_3` copy is pure duplicate junk: a separate, correctly defined `PowerTransformer` and pair of windings for `NL_TR2_3` already exists elsewhere in the file.
   - The `NL_TR2_4` copy is a fully orphaned fragment, that is to say there is no real `PowerTransformer` named `NL_TR2_4` anywhere else in the model.

2. **Dangling (broken) reference.** The second winding of `NL_TR2_3` references `TransformerEnd.BaseVoltage = _abb2348a-aa10-446f-9f5d-49f93f211534`, but the only 15.75 kV `BaseVoltage` object that actually exists in the file is `_abb2348a-aa10-446f-9f5d-49f93f211535`. The two IDs differ by only the last number (it should be 4 instead of 5). This looks like a copy-paste error or a typo and leaves that winding's base voltage unresolved.

3. **Missing mandatory attribute.** The generator `NL-G3` (SynchronousMachine `_1dc9afba-23b5-41a0-8540-b479ed8baf4b`) has no `RotatingMachine.ratedU` value (which can be seen in Task 3.4. as this specific generator is missing a voltage value in the results table above), while the model's other two machines, `NL-G1` and `NL-G2`, both correctly declare `ratedU = 15.75 kV`.

**Power-system / engineering errors** — the data is structurally valid but does not make physical sense:

4. **TATL lower than PATL (found in Task 3.3).** For line `NL-Line_5`, the temporarily allowed limit (TATL, 500 A) is lower than the permanently allowed limit (PATL, 1876 A). A TATL is a short-duration overload rating and should always be greater than or equal to the continuous (PATL) rating.

5. **Generator active-power rating inconsistent with its apparent power and power factor.** For `NL-G2` and `NL-G3`, `GeneratingUnit.maxOperatingP = 250 MW`, but `RotatingMachine.ratedS= 250 MVA` together with `RotatingMachine.ratedPowerFactor = 0.9` implies a maximum  deliverable active power of `250 × 0.9 = 225 MW` which matches `GeneratingUnit.nominalP  = 225 MW`. However, the declared `maxOperatingP` of 250 MW would only be achievable at a power factor of 1.0, contradicting the machine's own rated power factor of 0.9.


Side note: some of these errors were found by manual inspection of the XML file of the EQ profile and code was written to highlight them (in the case of the power factor mismatch). Other errors were found by writing automated checks that scanned the entire model for inconsistencies (such as the NL_TR2_3 BaseVoltage ID check ). The code for those checks is included in the notebook, and the results of those checks are summarized above.